In [1]:
import pprint
import urllib.parse
import json5
from qwen_agent.agents import Assistant
from qwen_agent.tools.base import BaseTool, register_tool
from qwen_agent.utils.output_beautify import typewriter_print

In [2]:
llm_cfg = {
    'model': 'Qwen/Qwen3-0.6B',
    'model_server': 'http://localhost:8000/v1',
    'api_key': 'EMPTY',

    # (Optional) LLM hyperparameters for generation:
    'generate_cfg': {
        "temperature": 0.6,
        "top_p": 0.95,
        "top_k": 20,
        "seed": 42
    }
}

In [3]:
from datasets import load_dataset

dataset = load_dataset("openai/gsm8k", "main", split="test")

dataset[0]

{'question': "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?",
 'answer': 'Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.\n#### 18'}

In [4]:
def extract_hash_answer(text):
    if "####" not in text:
        return None
    return text.split("####")[1].strip()
extract_hash_answer(dataset[0]["answer"])

'18'

In [5]:
dataset = dataset.map(lambda x: {
    "answer": extract_hash_answer(x["answer"])
})

In [6]:
tools = ['code_interpreter']  # `code_interpreter` is a built-in tool for executing code.
tools = []
bot = Assistant(llm=llm_cfg,
                # system_message=system_instruction,
                function_list=tools,
                )
                

In [7]:
dataset

Dataset({
    features: ['question', 'answer'],
    num_rows: 1319
})

In [8]:
math_prompt = "Please reason step by step, and put your final answer within \boxed{}."
tool_prompt = "You must use the code interpreter tool."

In [9]:
import asyncio
from copy import deepcopy

async def run_episode(agent, prompt):
    messages = [{"role": "user", "content": f"{prompt}"}]
    result = await asyncio.to_thread(agent.run_nonstream, messages=messages)
    return result

agents = None

async def evaluate_passk(prompt, k):
    global agents
    if not agents:
        agents = [None] * k
        for i in range(k):
            cfg = deepcopy(llm_cfg)
            cfg["generate_cfg"]["seed"] += i
            agents[i] = Assistant(llm=cfg, function_list=tools)

    results = await asyncio.gather(*[
        run_episode(agent, prompt)
        for agent in agents
    ])
    return results    

In [10]:
import re
def extract_boxed_answer(text):
    match = re.search(r"\$\s*\\boxed\{([^}]*)\}\s*\$", text)
    return match.group(1).strip() if match else None

In [11]:
import sympy as sp
def is_equivalent(ans1, ans2):
    if not ans1:
        return False
    try:
        expr1 = sp.sympify(ans1)
        expr2 = sp.sympify(ans2)
        return sp.simplify(expr1 - expr2) == 0
    except Exception:
        return ans1.strip() == ans2.strip()

In [ ]:
from tqdm import tqdm
import json

output_path = "./gsm8k_results_no_tool.jsonl"
k = 4
batch_size = 10

batch = []
pending = []

total_correct = 0

with open(output_path, "a", encoding="utf-8") as f:
    for i in tqdm(range(0, 200, batch_size)):
        batch = dataset.select(range(i, i+batch_size))
        for j, example in enumerate(batch):
            problem = example["question"]

            responses = await evaluate_passk(problem, k)
            k_correct = False
            for (tokens, num_tool_calls, tool_call_errors, response) in responses:

                # parse answer
                answer = response[-1]["content"]
                final_answer = extract_boxed_answer(answer)
                if final_answer is None and answer:
                    final_answer = answer.split()[-1]
                
                if is_equivalent(final_answer, example["answer"]):
                    correct = True
                    k_correct = True
                else:
                    correct = False

                pending.append({
                    "question_id" : i+j,
                    "question" : problem,
                    "response" : response,
                    "answer" : final_answer,
                    "tool_calls" : num_tool_calls,
                    "tool_call_errors" : tool_call_errors,
                    "correct" : correct,
                    "k_correct" : k_correct,
                    "tokens" : tokens,
                })

            if k_correct:
                total_correct += 1
        
        for x in pending:
            f.write(json.dumps(x) + "\n")
        f.flush()
        pending.clear()

100%|██████████| 3/3 [24:48<00:00, 496.16s/it]


In [49]:
result_dataset = load_dataset("json", data_files="gsm8k_results_tool_required.jsonl")["train"]
print(result_dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['id', 'question', 'response', 'answer', 'tool_calls', 'tool_call_errors', 'correct', 'k_correct', 'tokens'],
    num_rows: 800
})


In [117]:
result_dataset

Dataset({
    features: ['id', 'question', 'response', 'answer', 'tool_calls', 'tool_call_errors', 'correct', 'k_correct', 'tokens'],
    num_rows: 800
})

In [115]:
from huggingface_hub import HfApi

api = HfApi()
api.create_repo("simpissa/gsm8k-results", repo_type="dataset", private=False)


RepoUrl('https://huggingface.co/datasets/simpissa/gsm8k-results', endpoint='https://huggingface.co', repo_type='dataset', repo_id='simpissa/gsm8k-results')

In [116]:
result_dataset.push_to_hub("simpissa/gsm8k-results")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


CommitInfo(commit_url='https://huggingface.co/datasets/simpissa/gsm8k-results/commit/de79f80ade00c71124fe17843d5a15c2d44aa59e', commit_message='Upload dataset', commit_description='', oid='de79f80ade00c71124fe17843d5a15c2d44aa59e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/simpissa/gsm8k-results', endpoint='https://huggingface.co', repo_type='dataset', repo_id='simpissa/gsm8k-results'), pr_revision=None, pr_num=None)

In [50]:
df = result_dataset.to_pandas()
